
# mBART-50 (zh→en) — Baseline TF Training (Wuxia Domain)

**TFG – Baseline NMT (MarianMT)**  
This notebook trains and evaluates to model **MarianMT** (`Helsinki-NLP/opus-mt-zh-in`) with **TensorFlow** and **Hugging Face** using to dataset paralelo **wuxia** (Chinese→English) already prepared in format `datasets` (HF).



> **Requirements of the dataset**: directory HF Datasets with *splits* `train`, `validation`, `test` and columns `zh` (Chinese) and `en` (English):  
> `processed_data/wuxia_zh_en_clean/`


## 1) Environment of execution and installation of dependencies

In [ ]:


import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Name of the GPU:", torch.cuda.get_device_name(0))



In [ ]:
# Configuration of carpetas for entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent.parent.parent.parent
BASE_DIR.mkdir(exist_ok=True)

# Structure repository
for sub in ["evaluation", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Structure created (if not existed):")
for p in ["evaluation", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Score: the dataset must existir in: CORPUS/proccesed data/wuxia_zh_en_clean


## 2) Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Paths (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean" 
    output_dir: Path   = BASE_DIR / "models" / "mbart50_wuxia" / "checkpoint-104304"  
    
    ckpt_path: Path   =  BASE_DIR / "models" / "mbart50_wuxia" / "checkpoint-104304"                      
    training_dir: Path = BASE_DIR / "training"    

    evaluation_dir: Path = BASE_DIR / "evaluation" / "nmt"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate"
    
    translate_file: Path =  "mbart_10.txt"
    results_file: Path = "results.txt"

    # Columns of the dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Codes of language mBART-50
    src_lang: str = "zh_CN"
    tgt_lang: str = "en_XX"

    # Model
    model_ckpt: str = "facebook/mbart-large-50-many-to-many-mmt"

    # Training
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 5 # 600 MILLION d parametros + a run of 10 epochs failed by outage of power lead to this cpnclusion
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 2
    # Uses all the dataset in the execution final (ajusta if want depurar more fast)
    fraction: float = 1

cfg = Config()
print(cfg)


In [ ]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas for reproducibilidad
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # For reproducibilidad estricta (ligera penalty of rendimiento)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas and backend configured.")


## 4) Load dataset (Hugging Face Datasets)

In [ ]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"The dataset was not found at: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columns
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' must contener columns '{src_col}' y '{tgt_col}'. Columns: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# Subsampling optional for tests quick
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fraction={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


## 5) Load tokenizador and model mBART-50 (zh→en)

> **Score:** mBART-50 requiere indicar `src_lang` and `tgt_lang` (p.ej., `zh_CN` → `en_XX`). The `forced_bos_token_id` is establece for generate in the idioma destino.

In [ ]:
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
# Tokenizer
tokenizer = MBart50TokenizerFast.from_pretrained(cfg.model_ckpt)
# Idiomas
tokenizer.src_lang = cfg.src_lang
tokenizer.tgt_lang = cfg.tgt_lang

# Model (PyTorch) and transfer to the device

model = MBartForConditionalGeneration.from_pretrained(cfg.model_ckpt)
# Forzar idioma target in generation
model.config.forced_bos_token_id = tokenizer.lang_code_to_id[cfg.tgt_lang]
model.config.decoder_start_token_id = tokenizer.lang_code_to_id[cfg.tgt_lang]
model.to(device)

# Info useful
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {cfg.model_ckpt}")
print(f"Parameters total: {n_params:,}")

## 6) Preprocesamiento and tokenization

In [ ]:
# Function for tokenize examples
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples[cfg.src_col],
        max_length=cfg.max_source_length,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples[cfg.tgt_col],
        max_length=cfg.max_target_length,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# Apply tokenization to all the dataset
tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[0])


## 7) Data collator y `tf.data.Dataset`

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Data collator for mBART-50 with PyTorch
# Is encarga of align dynamically the sequences and create batches listos for the model
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    padding="longest",   # Can use "max_length" if want batches more uniformes
    return_tensors="pt"
)

# Example of batch
batch = data_collator([train_ds[i] for i in range(2)])
for k, v in batch.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")

## 8) Optimizador, callbacks and compilation

In [ ]:
## 8) Configuration of training — PyTorch + Seq2SeqTrainer

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Directory for results
run_dir = cfg.output_dir
run_dir.mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(run_dir),
    overwrite_output_dir=True,
    eval_strategy="epoch",                  # Evaluar to the final of each epoch
    save_strategy="epoch",  
    save_safetensors=True,                      # Save checkpoint by epoch
    save_total_limit=2,                      # Maximum. number of checkpoints saved
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    weight_decay=cfg.weight_decay,
    logging_dir=str(run_dir / "logs"),
    logging_strategy="steps",
    logging_steps=50,
    predict_with_generate=True,              # Generate sequences in validation
    fp16=False,
    bf16=True,
    torch_compile=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)
# Trainer for tareas Seq2Seq
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

print(" Seq2SeqTrainer configured (PyTorch).")


In [ ]:
# from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Usando dispositivo: {device}")


# # Path to your checkpoint
# ckpt_path: Path   = cfg.output_dir / "checkpoint-104304"

# # 1️ Load tokenizer (uses the files that already are inside of the checkpoint)
# tokenizer = MBart50TokenizerFast.from_pretrained(ckpt_path)
# tokenizer.src_lang = cfg.src_lang
# tokenizer.tgt_lang = cfg.tgt_lang

# # 2️ Load model with the weights trained
# model = MBartForConditionalGeneration.from_pretrained(ckpt_path).to(device)


In [ ]:
# --- test inline (optional, for debug) ---
probe_text = "只听后面蹄声急促,一骑马追来"
probe_inp = tokenizer(probe_text, return_tensors="pt").to(model.device)
probe_out = model.generate(probe_inp["input_ids"], num_beams=4, max_length=64)
print("Inline gen:", tokenizer.decode(probe_out[0], skip_special_tokens=True))

probe_text = "人女仙们还未来得及细看,毛十二就分别派发蛊虫"
probe_inp = tokenizer(probe_text, return_tensors="pt").to(model.device)
probe_out = model.generate(probe_inp["input_ids"], num_beams=4, max_length=64)
print("Inline gen:", tokenizer.decode(probe_out[0], skip_special_tokens=True))

[118061/130380 26:13:31 < 2:44:11, 1.25 it/s, Epoch 4.53/5]
Epoch	Training Loss	Validation Loss
1	1.273200	1.251087
2	1.117700	1.142141
3	0.974200	1.080684
4	0.845100	1.047662

## 10) Save model and tokenizador

In [ ]:
## 10) Load best checkpoint of the model trained (PyTorch)

from transformers import MBart50TokenizerFast, MBartForConditionalGeneration, AutoConfig
import torch, json, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Siempre use the directory final where saved weights + tokenizer
best_ckpt_path = getattr(trainer.state, "best_model_checkpoint", None)
if best_ckpt_path is None:
    best_ckpt_path = cfg.output_dir
    print("Was not found 'best_model_checkpoint'. Loading from output_dir final.")
else:
    print(f"Best checkpoint detectado: {best_ckpt_path}. Using output_dir final for load all.")
    best_ckpt_path = cfg.output_dir  # aseguramos consistencia

print(f"Loading from: {best_ckpt_path}")

# 1) Tokenizer loaded of the output_dir
tokenizer = MBart50TokenizerFast.from_pretrained(best_ckpt_path)
tokenizer.src_lang = cfg.src_lang
tokenizer.tgt_lang = cfg.tgt_lang

# 2) Config CLEANS of the checkpoint base (not the that remained in output_dir)
base_ckpt = cfg.model_ckpt  # "facebook/mbart-large-50-many-to-many-mmt"
config_clean = AutoConfig.from_pretrained(base_ckpt)

# 3) Model with YOUR weights trained + config cleans
model = MBartForConditionalGeneration.from_pretrained(
    best_ckpt_path,   # uses weights trained of tu output_dir
    config=config_clean
).to(device)

print(" Model (model) and tokenizer (tokenizer) listos.")



In [ ]:


# TEST RELOAD: forzamos language target in generate (equal that in inline)
probe_text = "只听后面蹄声急促,一骑马追来"
probe_inp = tokenizer(probe_text, return_tensors="pt").to(device)
probe_out = model.generate(
    **probe_inp,
    forced_bos_token_id=tokenizer.lang_code_to_id[cfg.tgt_lang]
)
print("Reload gen:", tokenizer.decode(probe_out[0], skip_special_tokens=True))

In [ ]:

import json
import matplotlib.pyplot as plt
from pathlib import Path

run_info_path = Path(cfg.output_dir) / "run_info.json"
if run_info_path.exists():
    with open(run_info_path, "r", encoding="utf-8") as f:
        run_info = json.load(f)
    print(" Information of the training:")
    for k, v in run_info.items():
        print(f"  {k}: {v}")
else:
    print(f" Was not found {run_info_path}")

metrics_path = Path(cfg.output_dir) / "train_results.json"
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        train_metrics = json.load(f)
    print("\n Metrics final of training:")
    for k, v in train_metrics.items():
        print(f"  {k}: {v}")
else:
    print(f" Was not found {metrics_path}")

log_history_path = Path(cfg.output_dir) / "trainer_state.json"
if log_history_path.exists():
    with open(log_history_path, "r", encoding="utf-8") as f:
        trainer_state = json.load(f)

    # Info of the best checkpoint
    best_ckpt = trainer_state.get("best_model_checkpoint", None)
    if best_ckpt:
        print(f"\n Mejor checkpoint: {best_ckpt}")
    else:
        print("\n Was not found information of the best checkpoint.")

    # Historial of metrics
    log_history = trainer_state.get("log_history", [])

    # Extraer metrics and pasos
    steps_train = [entry["step"] for entry in log_history if "loss" in entry]
    train_loss = [entry["loss"] for entry in log_history if "loss" in entry]

    steps_eval = [entry["step"] for entry in log_history if "eval_loss" in entry]
    eval_loss = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry]

    learning_rates = [entry["learning_rate"] for entry in log_history if "learning_rate" in entry]
    steps_lr = [entry["step"] for entry in log_history if "learning_rate" in entry]

    # Detectar metrics adicionales
    extra_metrics = {}
    for entry in log_history:
        for k, v in entry.items():
            if k.startswith("eval_") and k not in ["eval_loss"]:
                extra_metrics.setdefault(k, {"steps": [], "values": []})
                extra_metrics[k]["steps"].append(entry["step"])
                extra_metrics[k]["values"].append(v)

    # --- Chart loss ---
    plt.figure(figsize=(8,5))
    plt.plot(steps_train, train_loss, label="Train Loss")
    plt.plot(steps_eval, eval_loss, label="Eval Loss")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.title("Evolution of the Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    if learning_rates:
        plt.figure(figsize=(8,5))
        plt.plot(steps_lr, learning_rates, label="Learning Rate", color="orange")
        plt.xlabel("Steps")
        plt.ylabel("LR")
        plt.title("Evolution of the rate of learning")
        plt.legend()
        plt.grid(True)
        plt.show()

    for metric_name, data in extra_metrics.items():
        plt.figure(figsize=(8,5))
        plt.plot(data["steps"], data["values"], label=metric_name)
        plt.xlabel("Steps")
        plt.ylabel(metric_name)
        plt.title(f"Evolution of {metric_name}")
        plt.legend()
        plt.grid(True)
        plt.show()

else:
    print(f" Was not found {log_history_path}")

## 11) Evaluation (BLEU with SacreBLEU)

In [ ]:
## 11) Evaluation robusta + Metrics (SacreBLEU, chrF, TER, ROUGE-L, METEOR) — PyTorch

# Instalar dependencias (only if no the tienes)

from tqdm.auto import tqdm
import sacrebleu
from sacrebleu.metrics import CHRF, TER
from rouge_score import rouge_scorer
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import torch
import json
from bert_score import score as calc_bertscore
from comet import download_model, load_from_checkpoint

import time
start = time.time()

# Descargar recursos of NLTK (for METEOR)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# --- Parameters ---
EVAL_MAX_SAMPLES = 1000        # None = all the split
PRED_BEAMS = 4
BATCH_EVAL = max(1, cfg.batch_size // 2)

# --- Comprobaciones ---
assert 'model' in globals(), "Was not found `model`. Loads the model before."
assert 'tokenizer' in globals(), "Was not found `tokenizer`. Load it before."
assert 'val_ds' in globals() and 'test_ds' in globals(), "Faltan `val_ds` y/o `test_ds`."
assert 'cfg' in globals(), "Falta `cfg`."

# Ensure pad_token_id
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id


# --- Seleccionar split ---
eval_raw = test_ds if len(test_ds) > 0 else val_ds
n_total = len(eval_raw)
n_eval = n_total if (EVAL_MAX_SAMPLES is None) else min(n_total, int(EVAL_MAX_SAMPLES))
assert n_eval > 0, "No there are examples for evaluar."
def decode_ids_to_text(dataset, id_col):
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in dataset[id_col]
    ]

src_texts = decode_ids_to_text(eval_raw, "input_ids")[:n_eval]
ref_texts = decode_ids_to_text(eval_raw, "labels")[:n_eval]


# --- Generation by batches ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batched_generate(texts, batch_size=8, max_length=128, num_beams=4):
    preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=cfg.max_source_length
            ).to(device)
            outputs = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True,
                forced_bos_token_id=tokenizer.lang_code_to_id[cfg.tgt_lang]
            )
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

preds = batched_generate(
    src_texts,
    batch_size=BATCH_EVAL,
    max_length=cfg.max_target_length,
    num_beams=PRED_BEAMS
)

# --- Metrics ---
bleu_corpus = sacrebleu.corpus_bleu(preds, [ref_texts]).score

chrf_metric = CHRF(word_order=2)
chrf_corpus = chrf_metric.corpus_score(preds, [ref_texts]).score

ter_metric = TER()
ter_corpus = ter_metric.corpus_score(preds, [ref_texts]).score

def compute_rougeL_f1(hyp_list, ref_list):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    f1s = []
    for h, r in zip(hyp_list, ref_list):
        s = scorer.score(r, h)
        f1s.append(s['rougeL'].fmeasure)
    return float(np.mean(f1s)) * 100.0
rougeL_f1 = compute_rougeL_f1(preds, ref_texts)

def compute_meteor(hyp_list, ref_list):
    scores = []
    for hyp, ref in zip(hyp_list, ref_list):
        hyp_tok = wordpunct_tokenize(hyp) if isinstance(hyp, str) else hyp
        ref_tok = wordpunct_tokenize(ref) if isinstance(ref, str) else ref
        scores.append(meteor_score([ref_tok], hyp_tok))
    return float(np.mean(scores)) * 100.0
meteor_avg = compute_meteor(preds, ref_texts)


# --- BERTSCORE ---
print("Calculando BERTScore...")
# Extract the language target from cfg (if not it have configured therefore, change it by "in", "is", etc.)
tgt_language = getattr(cfg, "tgt_lang", "multilingual")
_, _, F1 = calc_bertscore(preds, ref_texts, lang=tgt_language, verbose=False)
bertscore_avg = float(F1.mean()) * 100.0

# --- COMET ---
print("Calculating COMET (can take a slightly in load the primera time)...")
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)
comet_model.eval()

# COMET needs the text original (src), the prediction (mt) and the reference (ref)
comet_data = [
    {"src": s, "mt": p, "ref": r} 
    for s, p, r in zip(src_texts, preds, ref_texts)
]
comet_output = comet_model.predict(comet_data, batch_size=BATCH_EVAL, progress_bar=True)
comet_avg = float(comet_output.system_score) * 100.0

end_time = time.time()

results = {
    "model" : getattr(cfg, "model_ckpt", "unknown_model"),
    "n_eval": n_eval,
    "num_beams": PRED_BEAMS,
    "batch_eval": BATCH_EVAL,
    "sacrebleu": round(bleu_corpus, 4),
    "chrf2": round(chrf_corpus, 4),
    "ter": round(ter_corpus, 4),
    "rougeL_f1": round(rougeL_f1, 4),
    "meteor": round(meteor_avg, 4), 
    "bertscore": round(bertscore_avg, 4),
    "comet": round(comet_avg, 4),
    "execution_time": round(end_time - start, 2)
}

if hasattr(cfg, "evaluation_dir"):
    os.makedirs(cfg.evaluation_dir, exist_ok=True)
    res_file = os.path.join(cfg.evaluation_dir, getattr(cfg, "results_file", "results.json"))
    with open(res_file, "a", encoding="utf-8") as f:
        f.write("\n")
        f.write(json.dumps(results, ensure_ascii=False, indent=4))

print("\n--- RESULTS FINAL ---")
print(json.dumps(results, indent=4))


In [ ]:
# tid = tokenizer.lang_code_to_id[cfg.tgt_lang]
# model.config.forced_bos_token_id = tid
# model.config.decoder_start_token_id = tid
# model.generation_config.forced_bos_token_id = tid

## 12) Sample cualitativa (n examples aleatorios)

In [ ]:
import random
import torch

# Mostrar predicciones aleatorias for inspection
n_show = 100
idxs = random.sample(range(len(eval_raw)), k=min(n_show, len(eval_raw)))

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in idxs:
    # Decode text source and reference from dataset tokenized
    zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
    en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

    # Tokenize input and move to device
    inputs = tokenizer(zh, return_tensors="pt").to(device)

    # Generate translation
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=cfg.max_target_length,
            num_beams=4,
            early_stopping=True,
            forced_bos_token_id=tokenizer.lang_code_to_id[cfg.tgt_lang]
        )

    en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

    print("="*80)
    print("ZH:", zh)
    print("IN (ref):", en_ref)
    print("IN (pred):", en_pred)


In [ ]:
# from tqdm import tqdm
# os.makedirs(cfg.translate_dir, exist_ok=True)
# translate_path = os.path.join(cfg.translate_dir, cfg.translate_file)

# with open(translate_path, "w", encoding="utf-8") as f:
#     for i in tqdm(range(len(eval_raw) // 10)):
#         zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
#         en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

#         # Tokenize input and move to device
#         inputs = tokenizer(zh, return_tensors="pt").to(device)

#         # Generate translation
#         with torch.no_grad():
#             out = model.generate(
#                 **inputs,
#                 max_length=cfg.max_target_length,
#                 num_beams=4,
#                 early_stopping=True,
#                 forced_bos_token_id=tokenizer.lang_code_to_id[cfg.tgt_lang]
#             )

#         en_pred = tokenizer.decode(out[0], skip_special_tokens=True)


#         # Save in the file
#         f.write("="*80 + "\n")
#         f.write("ZH: " + zh + "\n")
#         f.write("IN (ref): " + en_ref + "\n")
#         f.write("IN (pred): " + en_pred + "\n\n")


In [ ]:

# # ==============================================================================
# # CODE FOR EXTRAER METRICS OF DIFICULTAD (DATA CARTOGRAPHY)
# # Add to the final of each notebook
# # ==============================================================================

# import torch
# import pandas as pd
# import sacrebleu
# from tqdm.auto import tqdm
# from sacrebleu.metrics import CHRF
# # 1. Configuration
# # ----------------
# BATCH_SIZE = 64  # Increase it if have VRAM, decrease it if gives OOM
# OUTPUT_FILE = f"scores_{cfg.model_ckpt.replace('/', '_')}.csv"
# SRC_COL = cfg.src_col 
# TGT_COL = cfg.tgt_col

# print(f"--> Starting inference over the TRAIN SET for: {cfg.model_ckpt}")
# print(f"--> Output file: {OUTPUT_FILE}")

# # 2. Prepare Model
# # ------------------
# model.eval()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# # Ensure configuration of language for models multilingual (mBART, M2M100)
# if hasattr(tokenizer, "src_lang") and hasattr(cfg, "src_lang"):
#     tokenizer.src_lang = cfg.src_lang
# # For mBART/M2M100 forzamos the token of start if is necessary
# if hasattr(model.config, "forced_bos_token_id") and hasattr(tokenizer, "lang_code_to_id"):
#     try:
#         # Intentar obtain the ID of the language target
#         lang_id = tokenizer.lang_code_to_id.get(cfg.tgt_lang, None)
#         if lang_id is not None:
#             model.config.forced_bos_token_id = lang_id
#     except:
#         pass

# # 3. Loop of Inference over Train
# # ---------------------------------
# # Use raw_ds['train'] for have the text crudo and calculate BLEU real
# dataset = raw_ds['train'] 
# # dataset = dataset.select(range(1000)) # DESCOMENTAR FOR PROBAR RAPIDO

# results_indices = []
# results_scores = []

# # Iterate by batches
# for i in tqdm(range(0, len(dataset), BATCH_SIZE), desc="Generating"):
#     batch = dataset[i : i + BATCH_SIZE]
#     src_texts = batch[SRC_COL]
#     tgt_texts = batch[TGT_COL] # Referencias reales
    
#     # to) Tokenize
#     inputs = tokenizer(
#         src_texts, 
#         padding=True, 
#         truncation=True, 
#         max_length=cfg.max_source_length, 
#         return_tensors="pt"
#     ).to(device)
    
#     # b) Generate (Greedy Search for speed)
#     with torch.no_grad():
#         generated_ids = model.generate(
#             **inputs, 
#             max_new_tokens=cfg.max_target_length, 
#             num_beams=1, 
#             do_sample=False,
#             early_stopping=False
#         )
    
#     # c) Decode
#     decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
#     chrf_metric = CHRF(word_order=2)
#     # d) Calcular Sentence-chrf
#     for idx, (pred, ref) in enumerate(zip(decoded_preds, tgt_texts)):
        

#         score = sacrebleu.sentence_chrf(pred, [ref]).score        
#         # Save the index original of the dataset and the score
#         results_indices.append(i + idx)
#         results_scores.append(score)

# # 4. Save Results
# # ---------------------
# df_scores = pd.DataFrame({
#     'id': results_indices,
#     'chrf': results_scores
# })

# # Save in the directory of processed_data or where prefieras
# save_path = BASE_DIR / "processed_data" / OUTPUT_FILE
# df_scores.to_csv(save_path, index=False)
# print(f" Scores saved in: {save_path}")